# Simplifying the linear layer in partial rounds

In [ ]:
# Module Import
import numpy as np
import matplotlib.pyplot as plt


In [1]:
def is_mds(matrix):
    """Check if a matrix is MDS by checking if all square submatrices are non-singular."""
    # This is a simplified check for small matrices
    size = matrix.shape[0]
    # For our experiment, we focus on the branch number (diffusion)
    return True

def diffusion_test(matrix, name):
    size = matrix.shape[0]
    # Input: A "sparse" vector with only one active element
    input_vector = np.zeros(size)
    input_vector[0] = 1

    print(f"--- Testing Diffusion for: {name} ---")
    current_state = input_vector
    for i in range(1, 5):  # Simulate 4 rounds
        current_state = np.dot(matrix, current_state)
        # Count non-zero elements (using a small epsilon for float precision)
        active_elements = np.count_nonzero(np.abs(current_state) > 1e-10)
        print(f"Round {i}: Active elements = {active_elements}/{size}")
    print("\n")

# 1. Standard MDS Matrix (Simplified Cauchy Matrix)
# Poseidon1 style
size = 8
mds_matrix = n = np.fromfunction(lambda i, j: 1.0 / (i + j + 0.5), (size, size))

# 2. Non-MDS Matrix (Simplified Grain-like or Sparse Matrix)
# Poseidon2 style - much faster but potentially less diffusive
non_mds_matrix = np.eye(size)
for i in range(size):
    non_mds_matrix[i, (i+1)%size] = 1  # Sparse cyclic shift

diffusion_test(mds_matrix, "Poseidon1-style MDS Matrix")
diffusion_test(non_mds_matrix, "Simplified Non-MDS Matrix")

--- Testing Diffusion for: Poseidon1-style MDS Matrix ---
Round 1: Active elements = 8/8
Round 2: Active elements = 8/8
Round 3: Active elements = 8/8
Round 4: Active elements = 8/8


--- Testing Diffusion for: Simplified Non-MDS Matrix ---
Round 1: Active elements = 2/8
Round 2: Active elements = 3/8
Round 3: Active elements = 4/8
Round 4: Active elements = 5/8




In [11]:
# 1. Finite Field Configuration (using a large prime for simulation)
# In practice, Poseidon uses specific scalar fields (e.g., BLS12-381)
PRIME = 0x30644e72e131a029b85045b68181585d2833e84879b9709143e1f593f0000001

def s_box(x):
    """The non-linear layer: x^alpha (usually x^5 for Poseidon)"""
    return pow(int(x), 5, PRIME)

def apply_partial_round(state, matrix, constant):
    """
    Simulates a Partial Round in Poseidon.
    Only the first element goes through the S-box.
    """
    # Non-linear layer (only the first word)
    state[0] = s_box(state[0])

    # Add round constants
    state = (state + constant) % PRIME

    # Linear layer (Matrix multiplication)
    state = np.dot(matrix, state) % PRIME
    return state

# 2. Define Matrices
# Example MDS Matrix (Cauchy matrix style)
mds_matrix = np.array([
    [2, 3, 1],
    [3, 1, 2],
    [1, 2, 3]
], dtype=object)

# Example Non-MDS Matrix (Sparse or simpler structure)
non_mds_matrix = np.array([
    [1, 1, 0],
    [0, 1, 1],
    [1, 0, 1]
], dtype=object)

# 3. Test Diffusion (The Experiment)
def run_experiment(matrix, name):
    print(f"--- Testing {name} ---")
    # Initial state: only one active element (Sparse input)
    state = np.array([1, 0, 0], dtype=object)
    constant = np.array([7, 7, 7], dtype=object)
    state_history = []

    for r in range(5): # Simulate 5 partial rounds
        state = apply_partial_round(state, matrix, constant)
        state_history.append(state.copy())
        # Count non-zero elements (Diffusion measure)
        active_elements = np.count_nonzero(state)
        print(f"Round {r+1}: State = {state}, Active Words = {active_elements}")
    return state_history

# The original calls to run_experiment will be moved to a new cell to integrate with check_vulnerability.

In [13]:
non_mds_matrix = np.array([
    [1, 0, 0],
    [0, 1, 0],
    [0, 0, 1]
], dtype=object)

print("\n")
run_experiment(non_mds_matrix, "Non-MDS Matrix")



--- Testing Non-MDS Matrix ---
Round 1: State = [8 7 7], Active Words = 3
Round 2: State = [32775 14 14], Active Words = 3
Round 3: State = [37819301359644052734382 21 21], Active Words = 3
Round 4: State = [584596019920615470056902191945248499245941822600075379803746002408672118325
 28 28], Active Words = 3
Round 5: State = [3165560026526991798607995868737642229063830839804849594322118207582615527640
 35 35], Active Words = 3


[array([8, 7, 7], dtype=object),
 array([32775, 14, 14], dtype=object),
 array([37819301359644052734382, 21, 21], dtype=object),
 array([584596019920615470056902191945248499245941822600075379803746002408672118325,
        28, 28], dtype=object),
 array([3165560026526991798607995868737642229063830839804849594322118207582615527640,
        35, 35], dtype=object)]

In [14]:
# Check for Linear Relationship (Statistical Correlation)
def check_vulnerability(state_history):
    print("--- Vulnerability Analysis ---")
    for i in range(1, len(state_history)):
        diff = state_history[i][1] - state_history[i-1][1]
        print(f"Word 1 growth in Round {i}: {diff}")
        if diff == 7: # Assuming 7 is the constant we added
            print("CRITICAL: Linear leakage detected!")

In [15]:
# Run experiments and check for vulnerabilities
mds_history = run_experiment(mds_matrix, "MDS Matrix")
check_vulnerability(mds_history)

print("\n")

non_mds_history = run_experiment(non_mds_matrix, "Non-MDS Matrix")
check_vulnerability(non_mds_history)

--- Testing MDS Matrix ---
Round 1: State = [44 45 43], Active Words = 3
Round 2: State = [329832668 494748845 164916485], Active Words = 3
Round 3: State = [7807254461239536414316846990750510860006198
 11710881691859304621475270486125764640846561
 3903627230619768207158423495375256089668755], Active Words = 3
Round 4: State = [14571982792921991404439558826623127240936027402581531768036584709247810961162
 21857974189382987106659338239934651825131734906190226067819923311323518313136
 7285991396460995702219779413311579234976936180363594517712273855646450075770], Active Words = 3
Round 5: State = [229746069134716934699134559100366910679453064637320518869587775639661014218
 15037676626449219268836530686564158949740033117136124174583603747953636445683
 7370595748572066053982279187539182366799570035906725243927235996021906357506], Active Words = 3
--- Vulnerability Analysis ---
Word 1 growth in Round 1: 494748800
Word 1 growth in Round 2: 11710881691859304621475270486125764146097716
Word 1 g

In [16]:
# 1. Advanced Vulnerability Analysis Script
def analyze_diffusion_quality(matrix, name, rounds=10):
    print(f"=== Deep Analysis: {name} ===")
    state = np.array([1, 0, 0], dtype=object)
    constants = np.array([7, 7, 7], dtype=object)

    history = []
    for r in range(rounds):
        state = apply_partial_round(state, matrix, constants)
        history.append(state.copy())

    # Check for constant differences (Linearity Check)
    for word_idx in range(len(state)):
        diffs = [history[i][word_idx] - history[i-1][word_idx] for i in range(1, rounds)]
        # If all differences are the same, the growth is linear (BAD!)
        if all(d == diffs[0] for d in diffs):
            print(f"ALERT: Word {word_idx} exhibits LINEAR GROWTH! (Step: {diffs[0]})")
        else:
            print(f"PASS: Word {word_idx} exhibits non-linear complexity.")

# Run the automated check
analyze_diffusion_quality(non_mds_matrix, "Current Non-MDS Matrix")

=== Deep Analysis: Current Non-MDS Matrix ===
PASS: Word 0 exhibits non-linear complexity.
ALERT: Word 1 exhibits LINEAR GROWTH! (Step: 7)
ALERT: Word 2 exhibits LINEAR GROWTH! (Step: 7)


In [17]:
# Use the same PRIME and S-box logic
PRIME = 0x30644e72e131a029b85045b68181585d2833e84879b9709143e1f593f0000001

def s_box(x):
    return pow(int(x), 5, PRIME)

def apply_partial_round(state, matrix, constant):
    state[0] = s_box(state[0])
    state = (state + constant) % PRIME
    state = np.dot(matrix, state) % PRIME
    return state

# Non-MDS Matrix that caused linear growth
non_mds_matrix = np.array([
    [1, 0, 0],
    [0, 1, 0],
    [0, 0, 1]
], dtype=object)

constant = np.array([7, 7, 7], dtype=object)

# 2. Search for Partial Collision
def find_partial_collision():
    # Input A: [1, 5, 5]
    # Input B: [100, 5, 5] (Different first word, same others)
    input_a = np.array([1, 5, 5], dtype=object)
    input_b = np.array([100, 5, 5], dtype=object)

    state_a = input_a.copy()
    state_b = input_b.copy()

    print(f"Initial A: {state_a}")
    print(f"Initial B: {state_b}\n")

    for r in range(1, 6): # Run 5 rounds
        state_a = apply_partial_round(state_a, non_mds_matrix, constant)
        state_b = apply_partial_round(state_b, non_mds_matrix, constant)

        print(f"--- Round {r} ---")
        # Check if Word 1 and Word 2 are collided
        is_collided_w1 = (state_a[1] == state_b[1])
        is_collided_w2 = (state_a[2] == state_b[2])

        print(f"State A: {state_a}")
        print(f"State B: {state_b}")
        print(f"Collision on Word 1: {is_collided_w1}")
        print(f"Collision on Word 2: {is_collided_w2}\n")

find_partial_collision()

Initial A: [1 5 5]
Initial B: [100 5 5]

--- Round 1 ---
State A: [8 12 12]
State B: [10000000007 12 12]
Collision on Word 1: True
Collision on Word 2: True

--- Round 2 ---
State A: [32775 19 19]
State B: [100000000350000000490000000343000000120050000016814 19 19]
Collision on Word 1: True
Collision on Word 2: True

--- Round 3 ---
State A: [37819301359644052734382 26 26]
State B: [3828612029084315376402813639978897420597291949600502175786888916650343033023
 26 26]
Collision on Word 1: True
Collision on Word 2: True

--- Round 4 ---
State A: [584596019920615470056902191945248499245941822600075379803746002408672118325
 33 33]
State B: [10814900464992176632920508107278301662911845870644064693750211593741892309860
 33 33]
Collision on Word 1: True
Collision on Word 2: True

--- Round 5 ---
State A: [3165560026526991798607995868737642229063830839804849594322118207582615527640
 40 40]
State B: [5025971308332269870473758966234137447827698058870872608328948607734034041701
 40 40]
Collision o

# Comparative Verification

In [18]:
# 1. Define a simple transformation matrix T and its inverse
T = np.array([[1, 1, 0], [0, 1, 0], [0, 0, 1]], dtype=object)
T_inv = np.array([[1, -1, 0], [0, 1, 0], [0, 0, 1]], dtype=object)

# 2. Derive the "Equivalent" Non-MDS matrix from our original MDS
# M_equivalent = T * MDS * T_inv
mds_matrix = np.array([[2, 3, 1], [3, 1, 2], [1, 2, 3]], dtype=object)
equivalent_non_mds = np.dot(T, np.dot(mds_matrix, T_inv)) % PRIME

print("--- Equivalent Non-MDS Matrix (The Secure Way) ---")
print(equivalent_non_mds)


--- Equivalent Non-MDS Matrix (The Secure Way) ---
[[5
  21888242871839275222246405745257275088548364400416034343698204186575808495616
  3]
 [3
  21888242871839275222246405745257275088548364400416034343698204186575808495615
  2]
 [1 1 3]]
